为了完美模拟真实电商、量化环境下的高频输入，该实验不借用外部工具，直接用 PySpark纯内存算子和标准I/O，写一个可以持续向磁盘文件夹疯狂吐Parquet碎片的高频流式水源

In [0]:
import os
import time
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType

In [0]:
print(" === [INFINITE SOURCE GENERATOR] 独立无尽实时流发电厂全量开火 ===")
print(" 提示：本脚本会无限期循环发电。需要停止时，请手动点击当前单元格的 [Stop/Interrupt] 按钮。\n")

In [0]:
%sql
SHOW CATALOGS;

In [0]:
%sql
-- 创建 Schema
CREATE SCHEMA IF NOT EXISTS dbacademy.streaming;
-- 创建 Volume
CREATE VOLUME IF NOT EXISTS dbacademy.streaming.vol_yuto_stream_source;

**Volume 是什么？**（Databricks Unity Catalog 语境下）
在
 Databricks 中，Volume（卷） 是 Unity Catalog 提供的托管文件存储层，专门用来管理非表数据（如 CSV、JSON、Parquet、图片、日志、脚本等），可以把它理解为：一个安全、可共享、可版本控制的「云文件夹」，替代了传统 DBFS 的 /tmp、/FileStore 等路径。

In [0]:
print("=== [STREAMING SOURCE GENERATOR] 高频流式订单水源开始发电 ===\n")

# 定义数据在物理硬盘的安家地址
source_dir = "/Volumes/dbacademy/streaming/vol_yuto_stream_source/"

**前置说明：**
dbutils.fs 是 Databricks 内置文件操作工具，底层操作的是 DBFS（Databricks File System），兼容 S3/ADLS/GCS 等云存储路径；


**source_dir **是提前定义好的存储目录路径变量。
第一行：dbutils.fs.rm(source_dir, recurse=True)

**含义：**
删除 source_dir 这个目录：
rm = remove，删除文件 / 文件夹；

**第二个参数：** recurse=True：递归删除。如果目录里有子文件夹、所有文件，会全部一次性清空；不加 recurse=True 只能删空文件夹，有内容会报错。

**业务作用：**
清空目录里上次运行残留的数据、临时文件、旧分区，避免新旧数据混杂、重复读取。

**第二行：**
dbutils.fs.mkdirs(source_dir)

**含义：**
mkdirs = make directories，创建多级目录。区别于 mkdir：mkdir 只能创建单级文件夹；mkdirs 路径里多层不存在的父目录会一并建好。

**业务作用：**
删完旧目录后，重建一个全新、空的干净目录，作为本次程序的数据源目录。

In [0]:
dbutils.fs.rm(source_dir, recurse=True)
dbutils.fs.mkdirs(source_dir)

In [0]:
# create schema

order_schema = StructType([
    StructField("order_id", LongType(), True),
    StructField("client_id", LongType(), True),
    StructField("amount", DoubleType(), True),
    StructField("event_time", StringType(), True),
    StructField("status", StringType(), True)
])

# 核心模拟，启用高频无限循环，每两秒向文件夹吐出一个崭新的Micro-Batch.

order_id_counter = 100010
batch_id = 0

try:
    # 先生成 5 个批次进行测试对账（生产中可改为 while True 持续发电)
    while True:
        current_time = time.strftime("%Y-%m-%d %H:%M:%S", time.localtime())

        # 模拟当前微批次下的高频订单行（包含正常订单与极少数需要监控的异常大额/失败状态）
        mock_data = [
            (order_id_counter, 888, 1500.0, current_time, "SUCCESS"),
            (order_id_counter + 1, 999, 85000.0, current_time, "SUCCESS"), # 异常大额大单，测试后面监控的敏感度
            (order_id_counter + 2, 777, 299.0, current_time, "FAILED")      # 失败订单
        ]
        order_id_counter += 3
        batch_id += 1

        # 将当前批次转换为 DataFrame
        df_batch = spark.createDataFrame(mock_data,order_schema)

        # 物理落盘：将这个微批次作为独立的 Parquet 文件写进源头目录
        # 每一个文件的诞生，都代表着上游 Kafka 的一次推送
        file_path = f"{source_dir}batch_{batch_id}_{int(time.time())}.parquet"
        df_batch.write.mode("overwrite").parquet(file_path)


        print(f"[Batch: {batch_id}] 成功向流式水源目录吐出 3 条高频订单数据！时间: {current_time}")

        # 模拟真实的秒级/分钟级间歇：每吐完一个批次，歇气 2 秒
        time.sleep(2)

# 🛡️ 优雅熄火防线：当你点击 Stop 时，代码会在这里被安全拦截，绝不损坏当前正在写入的文件

except KeyboardInterrupt:
    print("\n🛑 [SHUTDOWN DIRECTION] 侦测到用户手动下达刹车指令！正在安全回收 Driver 线程...")
    print(" 最后一批次物理字节已成功落盘安全闭环。")



**补充解析上面的代码：**

**time.time()**是 Python 标准库 time 的函数：
返回 Unix 时间戳 —— 从 1970-01-01 00:00:00 UTC 到当前这一刻的总秒数（浮点数）。 因此时间戳是全局唯一标识，同一秒内运行才会重复，几乎不会重名；


用来给批量文件命名，区分不同批次、不同时间生成的数据文件，避免覆盖旧文件；
替代手动写日期，自动生成唯一后缀。

**parquet是什么呢？**

是一种文件格式，在spark中常见的文件格式有以下的类型：

#### 大数据初学者常用文件格式完整对比
除 CSV、JSON、Parquet 外，新手高频接触的 4 种格式：**Text、ORC、Avro、Delta（Delta Lake）**，附带适用场景、优缺点，贴合 Databricks/Spark 学习场景。

##### 1. Text（纯文本 .txt）
适用场景
- 原始日志采集（服务器日志、埋点日志）
- 无固定结构的纯文本、单行一条数据
- 最简单入门读取操作 `spark.read.text()`

优点
- 最轻量，无任何封装，任何编辑器直接打开
- 日志流式采集首选，兼容性拉满

缺点
- 无字段概念，需要手动写代码分割列（split）
- 无数据类型、无压缩，存储体积最大
- 不支持嵌套结构，查询效率极低

##### 2. ORC（Optimized Row Columnar，.orc）

适用场景
- Hive 生态传统数据仓库、离线批处理
- 海量结构化报表、分区大表存储

优点
- 列式存储，压缩率、查询速度略优于 Parquet
- Hive 原生深度适配，支持复杂索引、事务

缺点
- Spark/Databricks 生态普及度低于 Parquet
- 嵌套复杂 JSON 结构兼容性一般
- 跨语言读写支持弱于 Parquet

##### 3. Avro（.avro）

适用场景
- Kafka 流数据传输、消息队列持久化
- 跨语言数据交换（Java/Python/Go）
- 需要强 Schema 约束的实时数据流

优点
- 自带内置 Schema，数据和结构绑定，读写不用额外维护表结构
- 支持完整嵌套、数组结构，压缩优秀
- 实时流场景标准中间存储格式

缺点
- 列式查询性能不如 Parquet/ORC，适合传输不适合长期查询
- 人类无法直接打开查看内容

##### 4. Delta（Delta Lake，底层还是 parquet，.delta 目录）

适用场景Databricks 必学）
- Structured Streaming 流式读写、数据湖
- 需要更新、删除、事务、重复数据去重的场景
- 生产级数据仓库，解决批流一体问题

优点
- 基于 Parquet 封装，继承 Parquet 所有性能优势
- 支持 ACID 事务：可更新、删除、合并数据（普通 parquet 只能覆盖/追加）
- 支持数据版本回溯（时间旅行）、重复数据自动合并

缺点
- 只能在 Spark/Databricks 生态使用，其他引擎兼容差
- 元数据会产生额外小文件，小批量写入易出现文件碎片

---

#### 全格式汇总对比表（初学者速查）
| 格式 | 核心适用场景 | 核心缺点 |
|------|-------------|---------|
| CSV | 小数据、人工打开查看、导入导出 | 无数据类型、体积大、查询慢 |
| JSON | 半结构化嵌套埋点数据 | 压缩差，大数据查询性能差 |
| Text(txt) | 原始日志、无结构文本采集 | 无字段、需手动分割、性能最差 |
| Parquet | 数据湖批/流长期存储（行业通用标准） | 不能直接用记事本打开，不支持更新删除 |
| ORC | Hive 离线数仓、超大分区报表 | Spark生态普及低，跨语言支持弱 |
| Avro | Kafka 实时消息传输、跨语言交换 | 列式查询性能弱，适合传输不适合查询 |
| Delta Lake | Databricks 流式任务、支持更新/事务的数据湖 | 生态绑定Spark，存在元数据小文件开销 |

---

#### 新手选型简单建议（贴合你当前流式写入场景）
1. **本地调试、少量数据看内容**：CSV / JSON
2. **Kafka 实时消息接收**：Avro
3. **单纯批量读写、只追加不修改**：Parquet（你现在用的）
4. **流式任务、需要去重/更新/回滚数据**：Delta Lake（Databricks 生产首选）
5. **Hive 传统离线报表**：ORC
6. **原始日志采集**：Text txt
